In [ ]:
# ---- Install ONNX tooling (Internet must be ON) ----
!pip install -q onnx onnxruntime onnxscript


In [ ]:
# ---- Setup: find checkpoints, preprocess the test set once ----
import glob, json, os, time, zipfile
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch, timm, onnx, onnxruntime as ort
from PIL import Image
from sklearn.metrics import f1_score

IN, OUT = "/kaggle/input", "/kaggle/working"
ONNX_DIR = f"{OUT}/onnx"; os.makedirs(ONNX_DIR, exist_ok=True)
IMG = 224
N_THREADS = os.cpu_count()
print("onnxruntime", ort.__version__, "| torch", torch.__version__, "| CPU threads:", N_THREADS)

# Final checkpoints (MobileNet seed 1 = full-schedule rerun)
RUNS = {"mobilenetv3_small_100": ["0", "1_full", "2"], "mobilevit_xxs": ["0", "1", "2"]}
CKPT = {}
for model, seeds in RUNS.items():
    for s in seeds:
        hits = glob.glob(f"{IN}/**/{model}_final_seed{s}.pt", recursive=True)
        assert hits, f"{model}_final_seed{s}.pt not found - add the Phase 2 and rerun notebook outputs as inputs."
        CKPT[(model, s)] = hits[0]
        print(model, s, "->", hits[0])

def find_image_root(sample):
    if os.path.exists(os.path.join(IN, sample)):
        return IN
    first = sample.split("/")[0]
    for d in glob.glob(f"{IN}/**/{first}", recursive=True):
        if os.path.exists(os.path.join(os.path.dirname(d), sample)):
            return os.path.dirname(d)
    raise FileNotFoundError("Attach the PlantVillage dataset.")

df = pd.read_csv(glob.glob(f"{IN}/**/splits.csv", recursive=True)[0])
ROOT = find_image_root(df["path"].iloc[0])
CLASSES = df.drop_duplicates("label_idx").sort_values("label_idx")["label"].tolist()

def load_u8(paths):
    # Identical to Phase 2 eval transform before normalisation: PIL bilinear resize to 224x224
    def one(p):
        return np.asarray(Image.open(os.path.join(ROOT, p)).convert("RGB").resize((IMG, IMG), Image.BILINEAR))
    with ThreadPoolExecutor(N_THREADS) as ex:
        return np.stack(list(ex.map(one, paths)))

t0 = time.time()
test = df[df.split == "test"]
X_TEST, Y_TEST = load_u8(test["path"].tolist()), test["label_idx"].to_numpy()
calib = df[df.split == "train"].groupby("label_idx", group_keys=False).apply(
    lambda g: g.sample(min(len(g), 14), random_state=0))          # ~512 calibration images, all classes
X_CAL = load_u8(calib["path"].tolist())
print(f"test {X_TEST.shape}, calibration {X_CAL.shape}, loaded in {time.time() - t0:.0f}s")

def norm_stats(model):
    c = timm.models.get_pretrained_cfg(model).to_dict()
    return np.array(c["mean"], np.float32).reshape(1, 3, 1, 1), np.array(c["std"], np.float32).reshape(1, 3, 1, 1)

def to_input(u8, model):
    mean, std = norm_stats(model)
    return ((u8.astype(np.float32) / 255.0).transpose(0, 3, 1, 2) - mean) / std


In [ ]:
# ---- Export to ONNX, build FP16 / INT8 variants, evaluate every variant on the full test set ----
from onnxruntime.quantization import (CalibrationDataReader, QuantFormat, QuantType,
                                      quantize_dynamic, quantize_static)
from onnxruntime.quantization.shape_inference import quant_pre_process
from onnxruntime.transformers.float16 import convert_float_to_float16  # handles MobileViT's attention Casts


class CalibReader(CalibrationDataReader):
    def __init__(self, x, bs=16):
        self.batches = iter([{"input": x[i:i + bs]} for i in range(0, len(x), bs)])

    def get_next(self):
        return next(self.batches, None)


def session(path, threads):
    so = ort.SessionOptions()
    so.intra_op_num_threads = threads
    so.inter_op_num_threads = 1
    return ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])


def predict(path, x, bs=64):
    sess = session(path, N_THREADS)
    return np.concatenate([sess.run(None, {"input": x[i:i + bs]})[0].argmax(1) for i in range(0, len(x), bs)])


def export_onnx(model, path):
    dummy = torch.randn(1, 3, IMG, IMG)
    kw = dict(input_names=["input"], output_names=["logits"], opset_version=17,
              dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}})
    try:
        torch.onnx.export(model, dummy, path, dynamo=False, **kw)
    except TypeError:  # older torch without the dynamo argument
        torch.onnx.export(model, dummy, path, **kw)


def depthwise_convs(path):
    m = onnx.load(path)
    return [n.name for n in m.graph.node if n.op_type == "Conv"
            and any(a.name == "group" and a.i > 1 for a in n.attribute)]


rows = []
for (model_name, s), ckpt in CKPT.items():
    tag = f"{model_name}_seed{s[0]}"
    print(f"\n===== {tag} =====", flush=True)
    model = timm.create_model(model_name, pretrained=False, num_classes=len(CLASSES))
    model.load_state_dict(torch.load(ckpt, map_location="cpu"))
    model.eval()
    x_test, x_cal = to_input(X_TEST, model_name), to_input(X_CAL, model_name)

    paths = {"fp32": f"{ONNX_DIR}/{tag}_fp32.onnx"}
    export_onnx(model, paths["fp32"])
    with torch.no_grad():  # sanity check: ONNX matches PyTorch
        ref = model(torch.from_numpy(x_test[:8])).numpy()
    diff = np.abs(session(paths["fp32"], N_THREADS).run(None, {"input": x_test[:8]})[0] - ref).max()
    print(f"ONNX vs PyTorch max |logit diff| = {diff:.2e}")

    pre = f"{ONNX_DIR}/{tag}_fp32_pre.onnx"
    try:
        quant_pre_process(paths["fp32"], pre)  # shape inference + graph optimisation, recommended before quantization
    except Exception as e:
        print(f"  pre-processing failed ({type(e).__name__}: {e}); quantizing the plain FP32 model")
        pre = paths["fp32"]

    variants = {
        "fp16": lambda p: onnx.save(convert_float_to_float16(onnx.load(paths["fp32"]), keep_io_types=True), p),
        "int8_dynamic": lambda p: quantize_dynamic(pre, p, weight_type=QuantType.QInt8),
        "int8_static": lambda p: quantize_static(pre, p, CalibReader(x_cal), quant_format=QuantFormat.QDQ,
                                                 per_channel=True, activation_type=QuantType.QUInt8,
                                                 weight_type=QuantType.QInt8),
    }
    for v, build in variants.items():
        p = f"{ONNX_DIR}/{tag}_{v}.onnx"
        try:
            build(p); paths[v] = p
        except Exception as e:
            print(f"  {v}: build FAILED - {type(e).__name__}: {e}")
            rows.append({"model": model_name, "seed": int(s[0]), "variant": v, "error": f"build: {e}"})

    preds = {}
    for v, p in list(paths.items()):
        t0 = time.time()
        try:
            preds[v] = predict(p, x_test)
        except Exception as e:
            print(f"  {v}: inference FAILED - {type(e).__name__}: {e}")
            rows.append({"model": model_name, "seed": int(s[0]), "variant": v, "error": f"run: {e}"})
            continue
        acc = float((preds[v] == Y_TEST).mean())
        # If static INT8 collapses, retry with depthwise convolutions kept in float (known failure mode)
        if v == "int8_static" and "fp32" in preds and acc < float((preds["fp32"] == Y_TEST).mean()) - 0.02:
            dw = depthwise_convs(pre)
            print(f"  int8_static dropped to {acc:.4f}; retrying with {len(dw)} depthwise convs excluded")
            p2 = f"{ONNX_DIR}/{tag}_int8_static_nodw.onnx"
            try:
                quantize_static(pre, p2, CalibReader(x_cal), quant_format=QuantFormat.QDQ, per_channel=True,
                                activation_type=QuantType.QUInt8, weight_type=QuantType.QInt8, nodes_to_exclude=dw)
                paths["int8_static_nodw"] = p2
            except Exception as e:
                print(f"  int8_static_nodw: build FAILED - {type(e).__name__}: {e}")
        rows.append({"model": model_name, "seed": int(s[0]), "variant": v,
                     "size_MB": os.path.getsize(p) / 2**20, "test_acc": acc,
                     "test_macro_f1": float(f1_score(Y_TEST, preds[v], average="macro")),
                     "agree_with_fp32": float((preds[v] == preds["fp32"]).mean()) if "fp32" in preds else 1.0,
                     "eval_s": time.time() - t0})
        print(f"  {v:18s} {rows[-1]['size_MB']:6.2f} MB  acc {acc:.4f}  F1 {rows[-1]['test_macro_f1']:.4f}  "
              f"agree {rows[-1]['agree_with_fp32']:.4f}", flush=True)
    if "int8_static_nodw" in paths and "int8_static_nodw" not in preds:
        v, p = "int8_static_nodw", paths["int8_static_nodw"]
        preds[v] = predict(p, x_test); acc = float((preds[v] == Y_TEST).mean())
        rows.append({"model": model_name, "seed": int(s[0]), "variant": v, "size_MB": os.path.getsize(p) / 2**20,
                     "test_acc": acc, "test_macro_f1": float(f1_score(Y_TEST, preds[v], average="macro")),
                     "agree_with_fp32": float((preds[v] == preds["fp32"]).mean())})
        print(f"  {v:18s} {rows[-1]['size_MB']:6.2f} MB  acc {acc:.4f}  F1 {rows[-1]['test_macro_f1']:.4f}")
    np.savez(f"{OUT}/{tag}_quant_preds.npz", labels=Y_TEST, **preds)
    if pre != paths["fp32"]:
        os.remove(pre)

res = pd.DataFrame(rows)
res.to_csv(f"{OUT}/quant_results.csv", index=False)


In [ ]:
# ---- Summary: mean ± std over seeds, and change vs. FP32 ----
ok = res[res["error"].isna()] if "error" in res.columns else res
fp32 = ok[ok.variant == "fp32"].set_index(["model", "seed"])["test_acc"]
ok = ok.assign(delta_acc_pp=[100 * (a - fp32[(m, s)]) for m, s, a in zip(ok.model, ok.seed, ok.test_acc)])
summary = ok.groupby(["model", "variant"]).agg(
    size_MB=("size_MB", "mean"), acc_mean=("test_acc", "mean"), acc_std=("test_acc", "std"),
    f1_mean=("test_macro_f1", "mean"), delta_acc_pp=("delta_acc_pp", "mean"),
    agree_with_fp32=("agree_with_fp32", "mean"), n_seeds=("seed", "count")).round(4)
summary.to_csv(f"{OUT}/quant_summary.csv")
print(summary.to_string())
if "error" in res.columns and res["error"].notna().any():
    print("\nFailures:\n", res[res["error"].notna()][["model", "seed", "variant", "error"]].to_string())


In [ ]:
# ---- Reference latency on this Kaggle x86 CPU (NOT the target device - the Pi numbers go in the report) ----
def latency(path, threads, warmup=20, runs=100):
    sess = session(path, threads)
    x = to_input(X_TEST[:1], "mobilevit_xxs" if "mobilevit" in path else "mobilenetv3_small_100")
    for _ in range(warmup):
        sess.run(None, {"input": x})
    t = []
    for _ in range(runs):
        t0 = time.perf_counter(); sess.run(None, {"input": x}); t.append((time.perf_counter() - t0) * 1000)
    return float(np.median(t)), float(np.percentile(t, 95))

lat = []
for f in sorted(glob.glob(f"{ONNX_DIR}/*_seed0_*.onnx")):
    for th in (1, N_THREADS):
        try:
            med, p95 = latency(f, th)
        except Exception as e:
            print(f"{os.path.basename(f):45s} FAILED: {type(e).__name__}"); continue
        lat.append({"file": os.path.basename(f), "threads": th, "median_ms": med, "p95_ms": p95})
        print(f"{os.path.basename(f):45s} {th} thread(s): median {med:7.2f} ms  p95 {p95:7.2f} ms", flush=True)
pd.DataFrame(lat).to_csv(f"{OUT}/latency_kaggle_x86.csv", index=False)


In [ ]:
# ---- Bundle for the Raspberry Pi: seed-0 ONNX files + a small labelled test subset ----
sub = test.reset_index(drop=True).groupby("label_idx", group_keys=False).apply(
    lambda g: g.sample(min(len(g), 10), random_state=0)).index.to_numpy()   # 10 images per class
expected = {}
for f in sorted(glob.glob(f"{ONNX_DIR}/*_seed0_*.onnx")):
    m = "mobilevit_xxs" if "mobilevit" in f else "mobilenetv3_small_100"
    expected[os.path.basename(f)] = float((predict(f, to_input(X_TEST[sub], m)) == Y_TEST[sub]).mean())
meta = {"classes": CLASSES, "img_size": IMG, "expected_subset_acc_kaggle": expected,
        "normalisation": {m: {"mean": norm_stats(m)[0].ravel().tolist(), "std": norm_stats(m)[1].ravel().tolist()}
                          for m in RUNS}}
np.savez_compressed(f"{OUT}/pi_test_subset.npz", images_u8=X_TEST[sub], labels=Y_TEST[sub])
with open(f"{OUT}/pi_meta.json", "w") as fh:
    json.dump(meta, fh, indent=2)
with zipfile.ZipFile(f"{OUT}/pi_bundle.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{ONNX_DIR}/*_seed0_*.onnx"):
        z.write(f, f"onnx/{os.path.basename(f)}")
    z.write(f"{OUT}/pi_test_subset.npz", "pi_test_subset.npz")
    z.write(f"{OUT}/pi_meta.json", "pi_meta.json")
print(f"pi_bundle.zip: {os.path.getsize(f'{OUT}/pi_bundle.zip') / 2**20:.1f} MB, "
      f"{len(sub)} test images, {len(expected)} models")
